In [1]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import sys

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
DATA = ROOT / "data/processed/caltech_12weeks_20261111T143402Z_9b087d"

if not ROOT.is_dir() or not DATA.is_dir():
    raise FileNotFoundError(f"请核对远程项目目录：{ROOT}\n数据目录：{DATA}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report_dir = ROOT / "report"
log_dir = ROOT / "results/logs" / f"model_spec_{stamp}"
report_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=False)

model_text = r"""# 数学模型规格

状态：2.1 草稿，待输入接口核对与参数确认；尚未经过数值验证。

## 1. 范围与固定参数

研究对象为 Caltech 单站聚合连续功率充电模型。
车辆连接固定，不建模桩分配、V2G、建筑负荷、电池 SOC 或原网络相位约束。
50 kW 是实验容量，不是真实供电容量声明。

| 符号 | 定义 | 单位及来源 |
|---|---|---|
| Δt | 控制时段长度 | 0.25 h，阶段一冻结 |
| P_i^max | 车辆所连接设备的功率上限 | 6.656 kW，阶段一设备约定 |
| P_t^station | 站点聚合容量 | 基准 50 kW，实验情景 |
| η | 能量转换效率 | 1，电网侧建模约定 |
| c_t | 时段电价 | USD/kWh，冻结实验 TOU |
| α | 费用权重 | 基准 0.5，计划值 |
| C_ref | 费用归一化参考 | USD，正值，待确定并冻结 |
| P_ref | 日峰值归一化参考 | kW，正值，待确定并冻结 |
| ε_U | 第二层允许的总缺口增量 | 初始 1e-5 kWh，待数值核查 |
| ε_service | 服务完成评价容差 | 0.01 kWh，区别于求解容差 |

UTC 用于时间运算；America/Los_Angeles 用于电价和自然日归属。
不强制每个当地自然日具有 96 个时段。

## 2. 信息集与服务目标

### 严格主队列
到达时有效的总请求和预计离站固定用于在线调度；
后续修改仅作审计。评价目标为到达时有效总请求。

### 晚提交补充队列
有效输入在 modifiedAt 生效后才能使用；在控制时刻 k，
仅使用不晚于 k 的最新有效输入。
首次输入生效前不安排供电。
请求是会话累计总目标，不是本次新增请求电量。

补充队列事后评价目标定义为：
连接期间已生效的最后一条有效总请求，按阶段一去重与排序规则确定。
该目标供离线参考与事后评价使用，不能提前传给在线调度器。
同一会话的四方法使用相同评价目标。

设 Q_i(k) 为当前可见总请求，D_i(k) 为本次仿真实际累计交付：
R_i(k) = max(Q_i(k) - D_i(k), 0)。
无有效输入时 R 不作为可调度需求。

需求下调至已交付量以下时停止继续供电；
记录当次下调导致的不可逆超额 max(D_i(k)-Q_i(k),0)。
最终超额另按 max(D_i(final)-E_eval_i,0) 计算，
不累加历次更新的超额快照。

历史 kWhDelivered、doneChargingTime 不用于在线状态。
环境的真实离站只在发生后对在线控制可见。

## 3. 统一 LP 输入接口

I：本次模型的会话集合。
T：按 UTC 排列的完整 15 分钟时段集合。
D：T 涉及的站点当地自然日。
a_it：会话 i 在时段 t 是否允许计划供电，取 0 或 1。
R_i：本次模型需安排的剩余电量，非负。
b_d：自然日 d 已经实际发生的 EV 峰值，非负。
新自然日尚未执行时 b_d=0。

模型只消费显式传入的 R_i、a_it、设备上限、容量、电价、
日映射和已发生峰值，不自行读取完整未来事件文件。

## 4. 决策变量

p_it >= 0：会话 i 在时段 t 的电网侧功率，kW。
u_i >= 0：计划完整服务范围内的未满足电量，kWh。
z_d >= 0：自然日 d 的计划及已发生峰值上界，kW。

p 为连续变量，不设置整数或启停变量。

## 5. 约束

设备与连接：
0 <= p_it <= a_it * P_i^max。

站点容量：
sum_i p_it <= P_t^station。

需求与缺口守恒：
sum_t p_it * Δt + u_i = R_i。
0 <= u_i <= R_i。

日峰值：
z_d >= sum_i p_it，对属于自然日 d 的每个时段 t。
z_d >= b_d。

非负功率、需求等式及非负缺口共同保证计划总交付不超过 R_i。
所有时间求和必须覆盖该车本次完整计划服务范围。

## 6. 两层优化

第一层：
min U = sum_i u_i，获得 U_star。

第二层增加：
sum_i u_i <= U_star + ε_U。

随后最小化：
J = α * C/C_ref + (1-α) * sum_d(z_d/P_ref)，
C = sum_t c_t * Δt * sum_i p_it。

C 为计划电费，不含峰值代理项。
跨日使用日峰值之和，不能以全窗口单一最大值代替。
目标函数与全部约束均为线性。

只有第一层确认最优，才将 U_star 称为最优缺口并进入
有服务优先保证的第二层。两层均记录状态、目标、耗时。
不读取无有效解的变量值；失败结果保留并标明原因。
求解器具体接口、版本及容差设置随实现记录。

总缺口最小不代表逐车公平；第二层允许在总缺口约束下
重新分配个体缺口，必须输出逐车结果。

## 7. 静态参考与在线模型的区别

静态 LP 是完整信息离线参考：
使用真实到达、真实离站和预先固定的事后评价目标；
初始已交付量为零，覆盖选定区间全部会话及必要尾段。
供电从真实到达向上取整，到真实离站向下取整；
完整可用时段以外 a_it=0。
静态模型不模拟用户逐次提交过程，因此具有信息优势。

在线模型只包含当前已连接且已有有效输入的会话。
可用起点不早于当前控制时刻、接入与输入释放后的控制网格。
预计离站向下取整，形成完整时段的计划可用集合。
不足一个完整时段的情况不删除，保留缺口与原因。

在线执行遇到时段内真实离站，环境立即停止供电，
按实际在线时长积分；计划能量与实际交付分别保存。
此项执行验证属于阶段三。

## 8. 滚动窗口与终端

基础窗口 H=24 h。
实际末端为基础窗口末端与当前可见有效预计离站末端的较晚者。
因此当前可见车辆的剩余服务范围完整进入模型，
基础版不需要另设窗口外尾段能量变量。

不预测未来新到达车辆；延伸不使用真实未来离站信息。
延伸电价使用冻结的每日循环 TOU。
记录每次有效窗口长度。

预计离站已过但车辆仍在场且需求未完成时，
采用预先固定的滚动延长一个控制时段规则，
保留逾期标记，不用真实离站替换预计离站。
预计离站向下取整造成的最后不足一整段时间，
在预计离站尚未实际到达前不提前启用逾期规则。

跨日不清空累计交付；日峰值按各自然日独立累计。
本阶段定义规则，动态运行和信息隔离验收留在阶段三。

## 9. 归一化待决项

C_ref、P_ref 不得使用测试集统计确定。
计划优先采用开发集容量受限无序充电的日费用和日峰值
正参考值；该基线尚未实现，因此当前不虚构数值。

2.2 手算和合成验证使用明确给定的正参考值，
并标记为验证配置，不作为正式实验参考值。
正式取值、空车日处理与零中位数处理规则在开发/验证阶段
明确冻结，再进入正式测试。

## 10. 公式与实现对应关系

| 数学内容 | 预定实现职责 | 核查输出 |
|---|---|---|
| a_it 与时段、日映射 | 输入构建及校验 | 可用时段与日归属表 |
| p_it 上下界 | optimization_model.py | 设备、连接残差 |
| 站点容量 | optimization_model.py | 站点最大残差 |
| 需求等式与 u_i | optimization_model.py | 逐车能量残差与缺口 |
| z_d 与 b_d | optimization_model.py | 逐日负荷、峰值约束残差 |
| 两层求解与状态处理 | optimization_model.py | 两层状态、目标、耗时 |
| 静态完整信息输入 | experiments/run_static.py | 输入配置与逐车计划 |
| 手算及边界验证 | tests/ | 断言与验证记录 |

## 11. 验收设计

2–3 车、4–8 时段的可行与拥堵例子先手算，再与 LP 比较。
覆盖零需求、无车、无可用时段、紧迫离站、设备功率不足、
共享容量不足、等价电价和跨日峰值。
另用单车手算或枚举检查费用与峰值权衡。

检查：
- 第一层最优总缺口与手算一致。
- 第二层总缺口 <= U_star + ε_U，并报告实际增量。
- 设备、站点及连接约束最大正违规 <= 1e-6 kW。
- 需求守恒最大绝对残差 <= 1e-5 kWh。
- 实际计划能量不超过需求；容差内误差单独列明。
- 日峰值变量满足负荷与已发生峰值的下界。
- 同输入提高容量，不应增加第一层最优缺口。
- 电费和目标由导出计划独立重算。
- 无有效解、非最优状态不被静默当作成功。

功率容差、能量容差、第二层服务容差分别记录。
小例子通过后才接动态环境，不登记正式优化效果。
"""
model_text = model_text.strip() + "\n"

target = report_dir / "model.md"
if target.exists() and target.read_text(encoding="utf-8") != model_text:
    target = report_dir / f"model_draft_{stamp}.md"
target.write_text(model_text, encoding="utf-8")

# 仅检查结构和内容哈希，不输出会话原始记录。
csv_paths = [
    DATA / "sessions_with_split.csv",
    DATA / "user_updates.csv",
    DATA / "environment_truth.csv",
    DATA / "environment_events.csv",
    DATA / "input_release_events.csv",
    DATA / "grid_checks.csv",
    ROOT / "data/external/tariff.csv",
]
json_paths = [DATA / "information_policy.json"]
config_paths = [
    ROOT / "configs/station.yaml",
    ROOT / "configs/tariff.yaml",
    ROOT / "configs/splits.yaml",
]

checks = {
    "created_at_utc": stamp,
    "python": sys.version,
    "model_path": str(target.relative_to(ROOT)),
    "files": [],
}

for path in csv_paths + json_paths + config_paths:
    record = {
        "path": str(path.relative_to(ROOT)),
        "exists": path.is_file(),
    }
    if path.is_file():
        record["sha256"] = hashlib.sha256(path.read_bytes()).hexdigest()
        if path.suffix == ".csv":
            with path.open("r", encoding="utf-8-sig", newline="") as f:
                reader = csv.reader(f)
                record["columns"] = next(reader, [])
                record["rows"] = sum(1 for _ in reader)
        elif path.suffix == ".json":
            content = json.loads(path.read_text(encoding="utf-8-sig"))
            record["content"] = content
        else:
            record["content"] = path.read_text(encoding="utf-8-sig")
    checks["files"].append(record)

checks["processed_csv_inventory"] = sorted(
    p.name for p in DATA.glob("*.csv")
)
audit = log_dir / "input_schema_audit.json"
audit.write_text(
    json.dumps(checks, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("模型规格草稿：", target)
print("输入检查记录：", audit)
print("\n=== 实际输入结构与配置 ===")
print(json.dumps(checks, ensure_ascii=False, indent=2))
print("\n当前状态：2.1 进行中，待核对；尚未求解 LP。")

模型规格草稿： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/report/model.md
输入检查记录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/model_spec_20261112T065151Z/input_schema_audit.json

=== 实际输入结构与配置 ===
{
  "created_at_utc": "20261112T065151Z",
  "python": "3.11.18 (main, Oct 26 2026, 11:35:17) [GCC 15.3.0]",
  "model_path": "report/model.md",
  "files": [
    {
      "path": "data/processed/caltech_12weeks_20261111T143402Z_9b087d/sessions_with_split.csv",
      "exists": true,
      "sha256": "31e64878608779505051adc875da47defa90f9ddb785b45033ba33f13822db3c",
      "columns": [
        "sessionID",
        "stationID",
        "siteID",
        "clusterID",
        "connectionTime",
        "cohort",
        "initial_kWhRequested",
        "initial_requestedDeparture",
        "split"
      ],
      "rows": 2075
    },
    {
      "path": "data/processed/caltech_12weeks_20261111T143402Z_9b087d/user_updates.csv",
      "exists": true,
      "sha256": "506

In [3]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import re
import shutil
import uuid

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
module_path = ROOT / "src/optimization_model.py"

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN = ROOT / "results/logs" / f"static_lp_fix_{stamp}_{uuid.uuid4().hex[:6]}"
RUN.mkdir(parents=True, exist_ok=False)

old_source = module_path.read_text(encoding="utf-8")
new_source, replaced = re.subn(
    r"\bm\.load\b", "m.station_load", old_source
)
if replaced == 0 and "m.station_load" not in old_source:
    raise RuntimeError("代码与预期不符，未修改文件，请回传当前代码。")

compile(new_source, str(module_path), "exec")
shutil.copy2(module_path, RUN / "optimization_model_before.py")
module_path.write_text(new_source, encoding="utf-8")

# 从修复后的源码直接加载，避免 Notebook 继续使用旧函数。
spec = importlib.util.spec_from_file_location(
    f"ev_optimization_fixed_{uuid.uuid4().hex}", module_path
)
fixed_module = importlib.util.module_from_spec(spec)
exec(compile(new_source, str(module_path), "exec"), fixed_module.__dict__)

def smoke_case(demand_each):
    return {
        "dt": 0.25,
        "alpha": 0.5,
        "C_ref": 1.0,
        "P_ref": 2.0,
        "epsilon_U": 1e-5,
        "time_limit_seconds": 30,
        "slots": [
            {
                "label": f"t{t}",
                "day": "synthetic_day",
                "price": price,
                "capacity": 2.0,
            }
            for t, price in enumerate([0.10, 0.10, 0.40, 0.40])
        ],
        "sessions": [
            {
                "id": i,
                "demand": demand_each,
                "pmax": 2.0,
                "available": [0, 1, 2, 3],
            }
            for i in ["A", "B"]
        ],
    }

summary = []
for name, demand_each, expected_u, expected_cost, expected_peak in [
    ("feasible", 0.5, 0.0, 0.25, 1.0),
    ("congested", 1.5, 1.0, 0.50, 2.0),
]:
    result = fixed_module.solve_lp(smoke_case(demand_each), RUN / name)

    checks = {
        "first_stage_unmet": abs(
            result["U_star_kWh"] - expected_u
        ) <= 1e-6,
        "cost": abs(result["cost_USD"] - expected_cost) <= 2e-5,
        "peak": abs(
            result["peak_by_day_kW"]["synthetic_day"] - expected_peak
        ) <= 3e-5,
    }
    (RUN / name / "handcheck.json").write_text(
        json.dumps(checks, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    if not all(checks.values()):
        raise AssertionError(
            f"{name} 手算对照未通过：{checks}；结果见 {RUN / name}"
        )

    summary.append({
        "case": name,
        **{
            k: result[k]
            for k in [
                "status",
                "U_star_kWh",
                "U_stage2_kWh",
                "cost_USD",
                "peak_by_day_kW",
                "max_power_violation_kW",
                "max_energy_residual_kWh",
                "service_lock_violation_kWh",
                "total_seconds",
            ]
        },
        "handcheck": checks,
    })

(RUN / "smoke_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("已替换保留名称引用数：", replaced)
print("修复后的代码：", module_path)
print("本次日志目录：", RUN)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\n两个例子求解及手算对照通过；完整阶段验收仍待后续验证。")

已替换保留名称引用数： 4
修复后的代码： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/src/optimization_model.py
本次日志目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/static_lp_fix_20261112T065651Z_da25a2
[
  {
    "case": "feasible",
    "status": "passed",
    "U_star_kWh": 0.0,
    "U_stage2_kWh": 1e-05,
    "cost_USD": 0.24999750000000004,
    "peak_by_day_kW": {
      "synthetic_day": 0.9999900000000002
    },
    "max_power_violation_kW": 0.0,
    "max_energy_residual_kWh": 0.0,
    "service_lock_violation_kWh": 0.0,
    "total_seconds": 0.4801167859695852,
    "handcheck": {
      "first_stage_unmet": true,
      "cost": true,
      "peak": true
    }
  },
  {
    "case": "congested",
    "status": "passed",
    "U_star_kWh": 1.0,
    "U_stage2_kWh": 1.00001,
    "cost_USD": 0.49999750000000004,
    "peak_by_day_kW": {
      "synthetic_day": 1.9999900000000002
    },
    "max_power_violation_kW": 0.0,
    "max_energy_residual_kWh": 0.0,
    "service_lock_viol

In [4]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
import subprocess
import sys
import uuid

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN = ROOT / "results/logs" / f"model_validation_{stamp}_{uuid.uuid4().hex[:6]}"
RUN.mkdir(parents=True, exist_ok=False)

# ---------- 1. 补齐无车解析分支 ----------
model_path = ROOT / "src/optimization_model.py"
source = model_path.read_text(encoding="utf-8")

old_guard = '''        if not cars:
            raise ValueError("无车解析分支留待 2.3 补充，不调用求解器")
'''
anchor = "        pairs = [(i, t) for i in ids for t in available[i]]"

empty_branch = '''
        # Empty-fleet analytic branch: no solver invocation.
        if not cars:
            objective = (1-alpha) * sum(past.values()) / pref
            report.update({
                "status": "passed",
                "mode": "analytic_empty_fleet",
                "U_star_kWh": 0.0,
                "U_stage2_kWh": 0.0,
                "service_change_kWh": 0.0,
                "epsilon_U_kWh": eps,
                "cost_USD": 0.0,
                "objective": objective,
                "peak_by_day_kW": dict(past),
                "peak_variable_by_day_kW": dict(past),
                "max_power_violation_kW": 0.0,
                "max_energy_residual_kWh": 0.0,
                "max_overcharge_kWh": 0.0,
                "max_u_bound_violation_kWh": 0.0,
                "max_peak_violation_kW": 0.0,
                "service_lock_violation_kWh": 0.0,
                "cost_recompute_error_USD": 0.0,
                "objective_recompute_error": 0.0,
            })
            with (out / "plan.csv").open(
                "w", newline="", encoding="utf-8"
            ) as f:
                csv.writer(f).writerow([
                    "sessionID", "slot", "day", "power_kW", "energy_kWh"
                ])
            with (out / "sessions.csv").open(
                "w", newline="", encoding="utf-8"
            ) as f:
                csv.writer(f).writerow([
                    "sessionID", "target_kWh", "planned_kWh", "unmet_kWh"
                ])
            return report

'''

if "# Empty-fleet analytic branch:" not in source:
    if old_guard not in source or source.count(anchor) != 1:
        raise RuntimeError("模型文件与预期不符，未修改，请回传文件内容。")
    updated = source.replace(old_guard, "", 1)
    updated = updated.replace(anchor, empty_branch + anchor, 1)
    compile(updated, str(model_path), "exec")
    shutil.copy2(model_path, RUN / "optimization_model_before.py")
    model_path.write_text(updated, encoding="utf-8")

# ---------- 2. 保存独立验证脚本 ----------
test_source = r'''
"""合成静态 LP 验证；不读取 ACN 会话或测试集。"""
from pathlib import Path
from copy import deepcopy
from itertools import product
from types import SimpleNamespace
from unittest.mock import patch
import argparse
import csv
import json
import sys
import traceback

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
import optimization_model as opt

parser = argparse.ArgumentParser()
parser.add_argument("--output", required=True)
args = parser.parse_args()
OUT = Path(args.output)
OUT.mkdir(parents=True, exist_ok=False)

def case(demand=1.0):
    return {
        "dt": 0.25, "alpha": 0.5,
        "C_ref": 1.0, "P_ref": 2.0,
        "epsilon_U": 1e-5, "time_limit_seconds": 30,
        "slots": [
            {"label": f"t{t}", "day": "day1",
             "price": price, "capacity": 2.0}
            for t, price in enumerate([0.1, 0.1, 0.4, 0.4])
        ],
        "sessions": [
            {"id": "A", "demand": demand, "pmax": 2.0,
             "available": [0, 1, 2, 3]}
        ],
    }

def near(actual, expected, tol=3e-5):
    assert abs(actual-expected) <= tol, (actual, expected, tol)

def audit_exports(c, directory, r):
    """从导出的原始功率独立核算，不采用求解器内部表达式。"""
    with (directory / "plan.csv").open(
        encoding="utf-8", newline=""
    ) as f:
        rows = list(csv.DictReader(f))
    with (directory / "sessions.csv").open(
        encoding="utf-8", newline=""
    ) as f:
        vehicle_rows = list(csv.DictReader(f))

    cars = {v["id"]: v for v in c["sessions"]}
    labels = {s["label"]: t for t, s in enumerate(c["slots"])}
    loads = [0.0] * len(c["slots"])
    delivered = {i: 0.0 for i in cars}
    seen = set()
    max_power = 0.0
    for row in rows:
        i, t = row["sessionID"], labels[row["slot"]]
        assert i in cars and (i, t) not in seen
        seen.add((i, t))
        assert row["day"] == c["slots"][t]["day"]
        p = float(row["power_kW"])
        allowed = t in cars[i]["available"]
        limit = cars[i]["pmax"] if allowed else 0.0
        max_power = max(max_power, -p, p-limit)
        near(float(row["energy_kWh"]), p*c["dt"], 1e-10)
        loads[t] += p
        delivered[i] += p*c["dt"]

    assert len(seen) == len(cars)*len(c["slots"])
    for t, slot in enumerate(c["slots"]):
        max_power = max(max_power, loads[t]-slot["capacity"])
    assert max_power <= 1e-6

    vehicle_ids = [v["sessionID"] for v in vehicle_rows]
    assert len(vehicle_ids) == len(set(vehicle_ids))
    assert set(vehicle_ids) == set(cars)
    max_energy = 0.0
    total_u = 0.0
    for row in vehicle_rows:
        i = row["sessionID"]
        u = float(row["unmet_kWh"])
        near(float(row["target_kWh"]), cars[i]["demand"], 1e-10)
        near(float(row["planned_kWh"]), delivered[i], 1e-10)
        max_energy = max(
            max_energy, abs(delivered[i]+u-cars[i]["demand"])
        )
        assert -1e-5 <= u <= cars[i]["demand"]+1e-5
        assert delivered[i] <= cars[i]["demand"]+1e-5
        total_u += u
    assert max_energy <= 1e-5
    near(total_u, r["U_stage2_kWh"], 1e-8)
    assert total_u <= r["U_star_kWh"]+c["epsilon_U"]+1e-7

    cost = sum(
        loads[t]*s["price"]*c["dt"]
        for t, s in enumerate(c["slots"])
    )
    near(cost, r["cost_USD"], 1e-8)
    for day in dict.fromkeys(s["day"] for s in c["slots"]):
        peak = max(
            [c.get("past_peak", {}).get(day, 0)]
            + [loads[t] for t, s in enumerate(c["slots"])
               if s["day"] == day]
        )
        near(peak, r["peak_by_day_kW"][day], 1e-8)
        assert r["peak_variable_by_day_kW"][day] >= peak-1e-6

    return {
        "independent_power_violation_kW": max_power,
        "independent_energy_residual_kWh": max_energy,
    }

records = []
results = {}

def run(name, c, expected_u, expected_cost=None, expected_peak=None):
    directory = OUT / name
    r = opt.solve_lp(c, directory)
    assert r["status"] == "passed"
    near(r["U_star_kWh"], expected_u, 1e-6)
    independent = audit_exports(c, directory, r)
    if expected_cost is not None:
        near(r["cost_USD"], expected_cost)
    if expected_peak is not None:
        near(r["peak_by_day_kW"]["day1"], expected_peak)
    results[name] = r
    return {
        "U_star_kWh": r["U_star_kWh"],
        "cost_USD": r["cost_USD"],
        **independent,
    }

jobs = []

c = case(0)
jobs.append(("zero_demand", c, 0, 0, 0))

c = case(1)
c["sessions"][0]["available"] = []
jobs.append(("no_available_slots", c, 1, 0, 0))

c = case(1)
c["sessions"][0]["available"] = [0]
jobs.append(("urgent_departure", c, 0.5, 0.05, 2))

c = case(3)
for s in c["slots"]:
    s["capacity"] = 100
jobs.append(("device_power_limit", c, 1, 0.5, 2))

c = case(1)
for s in c["slots"]:
    s["capacity"] = 0
jobs.append(("zero_station_capacity", c, 1, 0, 0))

c = case(0.5)
for s in c["slots"]:
    s["price"] = 0.2
jobs.append(("equal_price_peak_minimization", c, 0, 0.1, 0.5))

c = case(0.5)
c["past_peak"] = {"day1": 2.0}
jobs.append(("already_realized_peak", c, 0, 0.05, 2))

c = case(0.5)
c["alpha"] = 0
c["past_peak"] = {"day1": 2.0, "day2": 0.0}
for t, s in enumerate(c["slots"]):
    s["day"] = "day1" if t < 2 else "day2"
    s["price"] = 0.2
jobs.append(("cross_day_peak_sum", c, 0, 0.1, 2))

c = case()
c["sessions"] = []
c["past_peak"] = {"day1": 3.0}
jobs.append(("empty_fleet", c, 0, 0, 3))

for capacity in [1.0, 2.0, 3.0]:
    c = case()
    c["sessions"] = [
        {"id": i, "demand": 1.5, "pmax": 2.0,
         "available": [0, 1, 2, 3]}
        for i in ["A", "B"]
    ]
    for s in c["slots"]:
        s["capacity"] = capacity
    jobs.append((f"capacity_{capacity:g}", c, 3-capacity, None, None))

# 单车纯费用目标：枚举 {0,1,2} kW 的全部 81 个计划。
# 此例需求为 1 kWh，枚举网格恰好包含连续 LP 的最优解。
c = case(1)
c["alpha"] = 1
for s, price in zip(c["slots"], [0.1, 0.2, 0.3, 0.4]):
    s["price"] = price
enumerated_cost = min(
    sum(p*s["price"]*c["dt"] for p, s in zip(powers, c["slots"]))
    for powers in product([0.0, 1.0, 2.0], repeat=4)
    if abs(sum(powers)*c["dt"]-1.0) < 1e-12
)
near(enumerated_cost, 0.15, 1e-12)
jobs.append(("single_car_enumeration", c, 0, enumerated_cost, None))

for name, c, u, cost, peak in jobs:
    try:
        details = run(name, deepcopy(c), u, cost, peak)
        # 跨日峰值之和应为 2 kW；不向第二天增加负荷。
        if name == "cross_day_peak_sum":
            near(results[name]["peak_by_day_kW"]["day2"], 0)
            near(results[name]["objective"], 1)
        if name == "empty_fleet":
            assert results[name]["mode"] == "analytic_empty_fleet"
            assert results[name]["stages"] == []
        records.append({"case": name, "status": "passed", **details})
    except Exception:
        records.append({
            "case": name, "status": "failed",
            "traceback": traceback.format_exc(),
        })

# 显式比较容量序列，而不只核对各自手算值。
try:
    us = [results[f"capacity_{v}"]["U_star_kWh"] for v in ["1", "2", "3"]]
    assert all(b <= a+1e-6 for a, b in zip(us, us[1:]))
    records.append({"case": "capacity_monotonicity", "status": "passed"})
except Exception:
    records.append({
        "case": "capacity_monotonicity", "status": "failed",
        "traceback": traceback.format_exc(),
    })

def expect_rejection(name, c, error_type, solver=None):
    directory = OUT / name
    try:
        if solver is None:
            opt.solve_lp(c, directory)
        else:
            with patch.object(opt.pyo, "SolverFactory", return_value=solver):
                opt.solve_lp(c, directory)
    except error_type:
        failure = json.loads((directory / "report.json").read_text())
        assert failure["status"] == "failed"
        assert not (directory / "plan.csv").exists()
        assert (directory / "error.txt").is_file()
        return
    raise AssertionError("预期失败未发生")

# 非法输入必须留下失败记录，不产生正常计划。
try:
    c = case(-1)
    expect_rejection("invalid_negative_demand", c, ValueError)
    records.append({"case": "invalid_negative_demand", "status": "passed"})
except Exception:
    records.append({
        "case": "invalid_negative_demand", "status": "failed",
        "traceback": traceback.format_exc(),
    })

# 注入超时状态，只验证失败处理；不是实际求解器超时性能实验。
class NonoptimalSolver:
    def __init__(self):
        self.options = {}
    def available(self, exception_flag=False):
        return True
    def solve(self, model, **kwargs):
        assert kwargs["load_solutions"] is False
        return SimpleNamespace(solver=SimpleNamespace(
            status="warning",
            termination_condition=opt.pyo.TerminationCondition.maxTimeLimit,
        ))

try:
    expect_rejection(
        "nonoptimal_solution_guard", case(), RuntimeError, NonoptimalSolver()
    )
    records.append({"case": "nonoptimal_solution_guard", "status": "passed"})
except Exception:
    records.append({
        "case": "nonoptimal_solution_guard", "status": "failed",
        "traceback": traceback.format_exc(),
    })

passed = sum(r["status"] == "passed" for r in records)
summary = {
    "status": "passed" if passed == len(records) else "failed",
    "passed": passed, "total": len(records), "cases": records,
}
(OUT / "validation_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

lines = [
    "# 静态 LP 合成验证记录", "",
    f"状态：{summary['status']}；通过 {passed}/{len(records)} 项。",
    f"证据目录：`{OUT.relative_to(ROOT)}`。", "",
    "验证使用合成输入及验证专用归一化参考值，不读取正式测试集。",
    "非最优状态检查使用注入状态，不是实际超时性能测量。",
    "本记录不代表真实数据入口、在线信息隔离或阶段二整体验收。", "",
    "| 检查 | 状态 |",
    "|---|---|",
]
lines += [f"| {r['case']} | {r['status']} |" for r in records]
text = "\n".join(lines) + "\n"
(OUT / "model_validation.md").write_text(text, encoding="utf-8")

# 始终保留本次报告；仅全通过时更新报告目录的当前版本。
if summary["status"] == "passed":
    target = ROOT / "report/model_validation.md"
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        (OUT / "model_validation_before.md").write_bytes(target.read_bytes())
    target.write_text(text, encoding="utf-8")

print(json.dumps(summary, ensure_ascii=False, indent=2))
sys.exit(0 if summary["status"] == "passed" else 1)
'''

test_path = ROOT / "tests/test_static_lp.py"
test_path.parent.mkdir(parents=True, exist_ok=True)
compile(test_source, str(test_path), "exec")
if test_path.exists():
    shutil.copy2(test_path, RUN / "test_static_lp_before.py")
test_path.write_text(test_source.lstrip(), encoding="utf-8")

# ---------- 3. 独立进程验证，避免依赖 Notebook 内存 ----------
command = [
    sys.executable,
    str(test_path),
    "--output", str(RUN / "validation"),
]
completed = subprocess.run(
    command,
    cwd=str(RUN),
    text=True,
    capture_output=True,
    timeout=180,
)
(RUN / "stdout_stderr.txt").write_text(
    completed.stdout + "\n--- STDERR ---\n" + completed.stderr,
    encoding="utf-8",
)
print("日志目录：", RUN)
print("独立进程退出码：", completed.returncode)
print(completed.stdout)
if completed.stderr:
    print("STDERR：\n", completed.stderr)

if completed.returncode != 0:
    raise RuntimeError("验证未全部通过，请回传以上输出。")

日志目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/model_validation_20261112T070046Z_0a7925
独立进程退出码： 1
{
  "status": "failed",
  "passed": 15,
  "total": 16,
  "cases": [
    {
      "case": "zero_demand",
      "status": "passed",
      "U_star_kWh": 0.0,
      "cost_USD": 0.0,
      "independent_power_violation_kW": 0.0,
      "independent_energy_residual_kWh": 0.0
    },
    {
      "case": "no_available_slots",
      "status": "passed",
      "U_star_kWh": 1.0,
      "cost_USD": 0.0,
      "independent_power_violation_kW": 0.0,
      "independent_energy_residual_kWh": 0.0
    },
    {
      "case": "urgent_departure",
      "status": "failed",
      "traceback": "Traceback (most recent call last):\n  File \"/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/tests/test_static_lp.py\", line 208, in <module>\n    details = run(name, deepcopy(c), u, cost, peak)\n              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/home/jovyan/work/EV_Rolling

RuntimeError: 验证未全部通过，请回传以上输出。

In [6]:
from pathlib import Path
from datetime import datetime, timezone
import json
import shutil
import subprocess
import sys
import uuid

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN = ROOT / "results/logs" / f"model_validation_fix_{stamp}_{uuid.uuid4().hex[:6]}"
RUN.mkdir(parents=True, exist_ok=False)

test_path = ROOT / "tests/test_static_lp.py"
source = test_path.read_text(encoding="utf-8")

old = 'jobs.append(("urgent_departure", c, 0.5, 0.05, 2))'
new = '''jobs.append((
    "urgent_departure", c, 0.5,
    0.1 * (0.5 - c["epsilon_U"]),
    2.0 - c["epsilon_U"] / c["dt"],
))'''

if old in source:
    updated = source.replace(old, new, 1)
elif new in source:
    updated = source
else:
    raise RuntimeError("未找到预期测试行，未修改文件。")

# 单独严格核对：该例第二层最优解会使用全部允许缺口增量。
anchor = "    results[name] = r\n"
strict_check = '''    # Urgent case: exact epsilon-aware hand calculation.
    if name == "urgent_departure":
        near(r["U_stage2_kWh"], 0.5 + c["epsilon_U"], 1e-8)
        near(
            r["peak_by_day_kW"]["day1"],
            2.0 - c["epsilon_U"] / c["dt"],
            1e-7,
        )
        near(r["cost_USD"], 0.1 * (0.5-c["epsilon_U"]), 1e-8)
'''

if "# Urgent case: exact epsilon-aware hand calculation." not in updated:
    if updated.count(anchor) != 1:
        raise RuntimeError("未找到唯一核对位置，未修改文件。")
    updated = updated.replace(anchor, strict_check + anchor, 1)

compile(updated, str(test_path), "exec")
shutil.copy2(test_path, RUN / "test_static_lp_before.py")
test_path.write_text(updated, encoding="utf-8")

(RUN / "correction.json").write_text(
    json.dumps({
        "reason": "紧迫离站测试须将能量容差除以可充电时长后换算为功率",
        "expected_peak_kW": 1.99996,
        "expected_stage2_unmet_kWh": 0.50001,
        "expected_cost_USD": 0.049999,
        "model_changed": False,
        "hard_constraint_tolerances_changed": False,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

completed = subprocess.run(
    [sys.executable, str(test_path), "--output", str(RUN / "validation")],
    cwd=str(RUN),
    text=True,
    capture_output=True,
    timeout=180,
)
(RUN / "stdout_stderr.txt").write_text(
    completed.stdout + "\n--- STDERR ---\n" + completed.stderr,
    encoding="utf-8",
)

print("本次日志目录：", RUN)
print("独立进程退出码：", completed.returncode)
print(completed.stdout)
if completed.stderr:
    print("STDERR：\n", completed.stderr)

if completed.returncode != 0:
    raise RuntimeError("仍有失败项，请回传以上输出。")

本次日志目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/model_validation_fix_20261112T070359Z_fc48f6
独立进程退出码： 0
{
  "status": "passed",
  "passed": 16,
  "total": 16,
  "cases": [
    {
      "case": "zero_demand",
      "status": "passed",
      "U_star_kWh": 0.0,
      "cost_USD": 0.0,
      "independent_power_violation_kW": 0.0,
      "independent_energy_residual_kWh": 0.0
    },
    {
      "case": "no_available_slots",
      "status": "passed",
      "U_star_kWh": 1.0,
      "cost_USD": 0.0,
      "independent_power_violation_kW": 0.0,
      "independent_energy_residual_kWh": 0.0
    },
    {
      "case": "urgent_departure",
      "status": "passed",
      "U_star_kWh": 0.5,
      "cost_USD": 0.04999900000000001,
      "independent_power_violation_kW": 0.0,
      "independent_energy_residual_kWh": 0.0
    },
    {
      "case": "device_power_limit",
      "status": "passed",
      "U_star_kWh": 1.0,
      "cost_USD": 0.4999975,
      "independent_power_vio

In [7]:
from pathlib import Path
from datetime import datetime, timezone
import shutil
import subprocess
import sys
import uuid

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN = ROOT / "results/logs" / f"static_data_validation_{stamp}_{uuid.uuid4().hex[:6]}"
RUN.mkdir(parents=True, exist_ok=False)

runner_source = r'''
"""开发集严格主队列的完整信息静态 LP 建模验证。"""
from pathlib import Path
from datetime import datetime, timezone
import argparse
import hashlib
import importlib.metadata
import json
import sys
import traceback

import numpy as np
import pandas as pd
import yaml

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from optimization_model import solve_lp

parser = argparse.ArgumentParser()
parser.add_argument("--output", required=True)
args = parser.parse_args()
OUT = Path(args.output)
OUT.mkdir(parents=True, exist_ok=False)

DATA = ROOT / "data/processed/caltech_12weeks_20261111T143402Z_9b087d"
summary = {
    "status": "running",
    "scope": "development_main_static_validation",
    "information": "offline_full_information",
    "normalization": {
        "C_ref_USD": 10.0,
        "P_ref_kW": 50.0,
        "purpose": "validation_only_not_formal_experiment",
    },
}

def aware(series, name):
    if series.isna().any():
        raise ValueError(f"{name} 存在缺失时间")
    if not series.map(lambda x: pd.Timestamp(x).tzinfo is not None).all():
        raise ValueError(f"{name} 存在无时区时间")
    return pd.to_datetime(series, utc=True, errors="raise")

def maxabs(values):
    return float(np.max(np.abs(np.asarray(values, dtype=float))))

try:
    files = {
        "sessions": DATA / "sessions_with_split.csv",
        "truth": DATA / "environment_truth.csv",
        "policy": DATA / "information_policy.json",
        "station": ROOT / "configs/station.yaml",
        "splits": ROOT / "configs/splits.yaml",
        "tariff_config": ROOT / "configs/tariff.yaml",
        "tariff": ROOT / "data/external/tariff.csv",
        "model_code": ROOT / "src/optimization_model.py",
        "runner_code": Path(__file__).resolve(),
    }
    manifest = {
        key: {
            "path": str(path.relative_to(ROOT)),
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        }
        for key, path in files.items()
    }
    (OUT / "input_manifest.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    versions = {
        name: importlib.metadata.version(name)
        for name in ["pyomo", "highspy", "pandas", "numpy", "PyYAML"]
    }
    summary["versions"] = versions
    summary["python"] = sys.version
    summary["created_at_utc"] = datetime.now(timezone.utc).isoformat()

    station = yaml.safe_load(files["station"].read_text(encoding="utf-8"))
    splits = yaml.safe_load(files["splits"].read_text(encoding="utf-8"))
    tariff_config = yaml.safe_load(
        files["tariff_config"].read_text(encoding="utf-8")
    )
    policy = json.loads(files["policy"].read_text(encoding="utf-8"))

    assert station["network_model"] == "aggregate_continuous"
    assert station["original_branch_constraints_enforced"] is False
    assert station["charging_efficiency"] == 1
    assert station["energy_basis"] == "grid_side"
    assert station["control_interval_minutes"] == 15
    assert station["station_capacity_kW"] == 50
    assert abs(station["evse_max_power_kW"] - 6.656) < 1e-12
    assert station["timezone"] == splits["timezone"] == tariff_config["timezone"]
    assert policy["main_at_arrival"] == "freeze_arrival_request_and_departure_estimate"

    sessions = pd.read_csv(files["sessions"], dtype=str)
    truth = pd.read_csv(
        files["truth"],
        usecols=["sessionID", "disconnectTime"],
        dtype=str,
    )
    assert sessions["sessionID"].notna().all()
    assert truth["sessionID"].notna().all()
    assert not sessions["sessionID"].duplicated().any()
    assert not truth["sessionID"].duplicated().any()
    assert set(sessions["sessionID"]).issubset(set(truth["sessionID"]))

    cohort_counts = sessions.groupby(["split", "cohort"]).size()
    summary["cohort_counts"] = {
        f"{a}/{b}": int(n) for (a, b), n in cohort_counts.items()
    }
    print("实际队列规模：", summary["cohort_counts"], flush=True)

    # 使用明确的队列标签；不按最终字段是否存在来猜测队列。
    chosen = sessions.loc[
        (sessions["split"] == "development")
        & (sessions["cohort"] == "main_at_arrival")
    ].copy()
    if len(chosen) != 64:
        raise ValueError(
            "开发集严格主队列与已冻结的 64 条不符，请核对上方实际队列标签和规模"
        )

    chosen = chosen.merge(
        truth, on="sessionID", how="left", validate="one_to_one"
    )
    for column in [
        "connectionTime", "disconnectTime", "initial_requestedDeparture"
    ]:
        chosen[column] = aware(chosen[column], column)
    chosen["target_kWh"] = pd.to_numeric(
        chosen["initial_kWhRequested"], errors="raise"
    )
    assert np.isfinite(chosen["target_kWh"]).all()
    assert (chosen["target_kWh"] > 0).all()

    bounds = splits["splits"]["development"]
    start = pd.Timestamp(bounds["start_utc_inclusive"])
    end = pd.Timestamp(bounds["end_utc_exclusive"])
    assert start.tzinfo is not None and end.tzinfo is not None
    start, end = start.tz_convert("UTC"), end.tz_convert("UTC")
    assert (chosen["connectionTime"] >= start).all()
    assert (chosen["connectionTime"] < end).all()
    assert (chosen["disconnectTime"] <= end).all()
    assert (chosen["disconnectTime"] > chosen["connectionTime"]).all()

    grid = pd.date_range(start, end, freq="15min", inclusive="left")
    assert len(grid) == 4704
    dt = 0.25
    capacity = float(station["station_capacity_kW"])
    pmax = float(station["evse_max_power_kW"])
    labels = [t.isoformat() for t in grid]
    days = grid.tz_convert(station["timezone"]).strftime("%Y-%m-%d").tolist()

    tariff = pd.read_csv(files["tariff"], dtype=str)
    tariff["start_utc"] = aware(tariff["start_utc"], "tariff.start_utc")
    assert not tariff["start_utc"].duplicated().any()
    tariff["price_USD_per_kWh"] = pd.to_numeric(
        tariff["price_USD_per_kWh"], errors="raise"
    )
    prices = tariff.set_index("start_utc")["price_USD_per_kWh"].reindex(grid)
    assert prices.notna().all()
    assert np.isfinite(prices).all() and (prices >= 0).all()

    # 完整信息静态参考：真实到达向上取整，真实离站向下取整。
    chosen = chosen.sort_values(
        ["connectionTime", "sessionID"], kind="stable"
    ).reset_index(drop=True)
    first = chosen["connectionTime"].dt.ceil("15min")
    last = chosen["disconnectTime"].dt.floor("15min")
    chosen["start_index"] = [int(grid.searchsorted(t)) for t in first]
    chosen["end_index"] = [int(grid.searchsorted(t)) for t in last]
    chosen["full_slots"] = (
        chosen["end_index"] - chosen["start_index"]
    ).clip(lower=0)
    chosen["device_only_shortfall_lower_bound_kWh"] = (
        chosen["target_kWh"] - chosen["full_slots"]*dt*pmax
    ).clip(lower=0)
    chosen.to_csv(OUT / "input_sessions.csv", index=False)

    case = {
        "dt": dt, "alpha": 0.5,
        "C_ref": 10.0, "P_ref": 50.0,
        "epsilon_U": 1e-5, "time_limit_seconds": 30,
        "slots": [
            {"label": label, "day": day,
             "price": float(price), "capacity": capacity}
            for label, day, price in zip(labels, days, prices)
        ],
        "sessions": [
            {
                "id": row.sessionID,
                "demand": float(row.target_kWh),
                "pmax": pmax,
                "available": list(range(row.start_index, row.end_index)),
            }
            for row in chosen.itertuples(index=False)
        ],
    }
    result = solve_lp(case, OUT / "lp")

    # ---------- 从导出文件独立核验 ----------
    plan = pd.read_csv(
        OUT / "lp/plan.csv",
        dtype={"sessionID": str, "slot": str, "day": str},
    )
    vehicle = pd.read_csv(
        OUT / "lp/sessions.csv", dtype={"sessionID": str}
    ).set_index("sessionID")
    ids = chosen["sessionID"].tolist()
    assert set(plan["sessionID"]) == set(ids)
    assert set(plan["slot"]) == set(labels)
    assert len(plan) == len(ids)*len(grid)
    assert not plan.duplicated(["sessionID", "slot"]).any()
    assert not vehicle.index.duplicated().any()
    assert set(vehicle.index) == set(ids)

    index_by_label = dict(zip(labels, range(len(grid))))
    day_by_label = dict(zip(labels, days))
    tindex = plan["slot"].map(index_by_label)
    assert (plan["day"] == plan["slot"].map(day_by_label)).all()
    by_id = chosen.set_index("sessionID")
    allowed = (
        (tindex >= plan["sessionID"].map(by_id["start_index"]))
        & (tindex < plan["sessionID"].map(by_id["end_index"]))
    )
    limits = np.where(allowed, pmax, 0.0)
    power = plan["power_kW"].to_numpy()
    assert np.isfinite(power).all()
    assert np.isfinite(plan["energy_kWh"]).all()

    loads = plan.groupby("slot")["power_kW"].sum().reindex(labels)
    energy = plan.groupby("sessionID")["energy_kWh"].sum().reindex(ids)
    target = by_id.loc[ids, "target_kWh"]
    unmet = vehicle.loc[ids, "unmet_kWh"]
    assert np.isfinite(unmet).all()

    power_violation = float(max(
        0.0, np.max(-power), np.max(power-limits),
        (loads-capacity).max(),
    ))
    energy_residual = maxabs(energy + unmet - target)
    energy_integral_error = maxabs(plan["energy_kWh"] - power*dt)
    overcharge = float(max(0.0, (energy-target).max()))
    unmet_bound_violation = float(max(
        0.0, (-unmet).max(), (unmet-target).max()
    ))
    exported_session_error = max(
        maxabs(vehicle.loc[ids, "planned_kWh"]-energy),
        maxabs(vehicle.loc[ids, "target_kWh"]-target),
    )
    cost = float(np.dot(loads.to_numpy(), prices.to_numpy()) * dt)
    cost_error = abs(cost-result["cost_USD"])
    lock_violation = max(
        0.0, float(unmet.sum())-result["U_star_kWh"]-case["epsilon_U"]
    )
    lower_bound = float(
        chosen["device_only_shortfall_lower_bound_kWh"].sum()
    )

    daily = pd.DataFrame({
        "local_date": days,
        "load_kW": loads.to_numpy(),
        "cost_USD": loads.to_numpy()*prices.to_numpy()*dt,
    }).groupby("local_date").agg(
        peak_kW=("load_kW", "max"), cost_USD=("cost_USD", "sum")
    )
    peak_error = max(
        abs(row.peak_kW-result["peak_by_day_kW"][day])
        for day, row in daily.iterrows()
    )
    peak_variable_violation = max(
        [0.0] + [
            row.peak_kW-result["peak_variable_by_day_kW"][day]
            for day, row in daily.iterrows()
        ]
    )
    daily.to_csv(OUT / "daily_planned_metrics.csv")

    summary.update({
        "sessions": len(ids),
        "slots": len(grid),
        "zero_full_slot_sessions": int((chosen["full_slots"] == 0).sum()),
        "truth_rows_outside_formal_session_ids": len(
            set(truth["sessionID"])-set(sessions["sessionID"])
        ),
        "target_kWh": float(target.sum()),
        "planned_kWh": float(energy.sum()),
        "U_star_kWh": result["U_star_kWh"],
        "U_stage2_kWh": float(unmet.sum()),
        "device_only_shortfall_lower_bound_kWh": lower_bound,
        "planned_cost_USD": cost,
        "planned_max_peak_kW": float(daily["peak_kW"].max()),
        "independent_power_violation_kW": power_violation,
        "independent_energy_residual_kWh": energy_residual,
        "energy_integral_error_kWh": energy_integral_error,
        "overcharge_kWh": overcharge,
        "unmet_bound_violation_kWh": unmet_bound_violation,
        "exported_session_error_kWh": exported_session_error,
        "cost_recompute_error_USD": cost_error,
        "daily_peak_recompute_error_kW": peak_error,
        "peak_variable_violation_kW": peak_variable_violation,
        "service_lock_violation_kWh": lock_violation,
        "solver_stages": result["stages"],
    })
    assert power_violation <= 1e-6
    assert energy_residual <= 1e-5
    assert energy_integral_error <= 1e-8
    assert overcharge <= 1e-5 and unmet_bound_violation <= 1e-5
    assert exported_session_error <= 1e-5
    assert cost_error <= 1e-7
    assert peak_error <= 1e-6 and peak_variable_violation <= 1e-6
    assert lock_violation <= 1e-7
    assert result["U_star_kWh"] >= lower_bound-1e-5
    summary["status"] = "passed"
except Exception:
    summary["status"] = "failed"
    summary["traceback"] = traceback.format_exc()
finally:
    (OUT / "static_validation_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)

sys.exit(0 if summary["status"] == "passed" else 1)
'''

runner = ROOT / "experiments/run_static.py"
runner.parent.mkdir(parents=True, exist_ok=True)
compile(runner_source, str(runner), "exec")
if runner.exists():
    shutil.copy2(runner, RUN / "run_static_before.py")
runner.write_text(runner_source.lstrip(), encoding="utf-8")

with (RUN / "stdout_stderr.txt").open("w", encoding="utf-8") as log:
    completed = subprocess.run(
        [sys.executable, str(runner), "--output", str(RUN / "validation")],
        cwd=str(RUN),
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=180,
    )

print("独立入口：", runner)
print("本次日志目录：", RUN)
print("独立进程退出码：", completed.returncode)
print((RUN / "stdout_stderr.txt").read_text(encoding="utf-8"))
if completed.returncode != 0:
    raise RuntimeError("真实数据静态验证未通过，请回传以上完整输出。")

独立入口： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/experiments/run_static.py
本次日志目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/static_data_validation_20261112T070933Z_56abcc
独立进程退出码： 0
实际队列规模： {'development/main_at_arrival': 64, 'development/supplement_after_submission': 1227, 'test/main_at_arrival': 39, 'test/supplement_after_submission': 468, 'validation/main_at_arrival': 19, 'validation/supplement_after_submission': 258}
{
  "status": "passed",
  "scope": "development_main_static_validation",
  "information": "offline_full_information",
  "normalization": {
    "C_ref_USD": 10.0,
    "P_ref_kW": 50.0,
    "purpose": "validation_only_not_formal_experiment"
  },
  "versions": {
    "pyomo": "6.10.1",
    "highspy": "1.15.1",
    "pandas": "2.2.3",
    "numpy": "1.26.4",
    "PyYAML": "6.0.3"
  },
  "python": "3.11.18 (main, Oct 26 2026, 11:35:17) [GCC 15.3.0]",
  "created_at_utc": "2026-11-12T07:09:36.088538+00:00",
  "cohort_counts": {
    

In [8]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import shutil
import uuid

import numpy as np
import pandas as pd

ROOT = Path("/home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization")
DATA = ROOT / "data/processed/caltech_12weeks_20261111T143402Z_9b087d"
SYN = ROOT / "results/logs/model_validation_fix_20261112T070359Z_fc48f6/validation"
REAL = ROOT / "results/logs/static_data_validation_20261112T070933Z_56abcc/validation"

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = ROOT / "results/logs" / f"stage2_acceptance_{stamp}_{uuid.uuid4().hex[:6]}"
OUT.mkdir(parents=True, exist_ok=False)
print("本次审计目录：", OUT)

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def aware(series, name):
    assert series.notna().all(), f"{name} 有缺失"
    assert series.map(
        lambda x: pd.Timestamp(x).tzinfo is not None
    ).all(), f"{name} 含无时区时间"
    return pd.to_datetime(series, utc=True, errors="raise")

synthetic = read_json(SYN / "validation_summary.json")
real = read_json(REAL / "static_validation_summary.json")
assert synthetic["status"] == "passed"
assert synthetic["passed"] == synthetic["total"] == 16
assert real["status"] == "passed"

# 确认真实数据验证后，其输入文件与核心代码未变化。
manifest = read_json(REAL / "input_manifest.json")
for item in manifest.values():
    path = ROOT / item["path"]
    assert hashlib.sha256(path.read_bytes()).hexdigest() == item["sha256"], path

# ---------- 1. 逐车缺口归因 ----------
case = read_json(REAL / "lp/input.json")
inputs = pd.read_csv(REAL / "input_sessions.csv", dtype={"sessionID": str})
vehicles = pd.read_csv(REAL / "lp/sessions.csv", dtype={"sessionID": str})
detail = inputs.merge(vehicles, on="sessionID", validate="one_to_one")
assert len(detail) == 64

pmax = {c["id"]: c["pmax"] for c in case["sessions"]}
detail["pmax_kW"] = detail["sessionID"].map(pmax)
assert detail["pmax_kW"].notna().all()
assert np.allclose(detail["target_kWh_x"], detail["target_kWh_y"],
                   atol=1e-8, rtol=0)
detail["target_kWh"] = detail["target_kWh_x"]

arrival = aware(detail["connectionTime"], "connectionTime")
departure = aware(detail["disconnectTime"], "disconnectTime")
duration_h = (departure-arrival).dt.total_seconds()/3600

detail["continuous_device_lower_bound_kWh"] = (
    detail["target_kWh"] - detail["pmax_kW"]*duration_h
).clip(lower=0)
detail["grid_device_lower_bound_kWh"] = (
    detail["target_kWh"]
    - detail["pmax_kW"]*detail["full_slots"]*case["dt"]
).clip(lower=0)
detail["extra_lower_bound_from_grid_kWh"] = (
    detail["grid_device_lower_bound_kWh"]
    - detail["continuous_device_lower_bound_kWh"]
).clip(lower=0)
detail["stage2_gap_above_device_bound_kWh"] = (
    detail["unmet_kWh"]-detail["grid_device_lower_bound_kWh"]
)
assert (
    detail["stage2_gap_above_device_bound_kWh"] >= -1e-5
).all()

grid_bound = float(detail["grid_device_lower_bound_kWh"].sum())
assert abs(grid_bound-real["U_star_kWh"]) <= 1e-7
assert abs(
    detail["unmet_kWh"].sum()
    - grid_bound - case["epsilon_U"]
) <= 1e-7

detail["reason"] = np.select(
    [
        detail["full_slots"] == 0,
        detail["grid_device_lower_bound_kWh"] > 1e-5,
    ],
    [
        "no_full_slot_in_actual_connection",
        "device_limit_and_available_duration",
    ],
    default="no_structural_device_shortfall",
)
detail.to_csv(OUT / "session_shortfall_audit.csv", index=False)

# ---------- 2. 在开发集核对两类队列的需求口径 ----------
sessions = pd.read_csv(DATA / "sessions_with_split.csv", dtype=str)
truth = pd.read_csv(
    DATA / "environment_truth.csv",
    usecols=["sessionID", "disconnectTime"], dtype=str,
)
dev = sessions.loc[sessions["split"] == "development"].merge(
    truth, on="sessionID", how="left", validate="one_to_one"
)
assert len(dev) == 64+1227
for col in ["connectionTime", "disconnectTime"]:
    dev[col] = aware(dev[col], col)

updates = pd.read_csv(DATA / "user_updates.csv", dtype=str)
updates = updates.loc[updates["sessionID"].isin(dev["sessionID"])].copy()
flags = updates["valid"].str.strip().str.lower()
assert flags.isin(["true", "false", "1", "0"]).all()
updates = updates.loc[flags.isin(["true", "1"])].copy()

updates["modifiedAt"] = aware(updates["modifiedAt"], "modifiedAt")
updates["requestedDeparture"] = aware(
    updates["requestedDeparture"], "requestedDeparture"
)
updates["kWhRequested"] = pd.to_numeric(
    updates["kWhRequested"], errors="raise"
)
updates["input_order"] = pd.to_numeric(
    updates["input_order"], errors="raise"
)
assert np.isfinite(updates["kWhRequested"]).all()
assert (updates["kWhRequested"] > 0).all()
assert np.isfinite(updates["input_order"]).all()
assert (updates["input_order"] % 1 == 0).all()
assert not updates.duplicated(
    ["sessionID", "modifiedAt", "input_order"]
).any()

updates = updates.merge(
    dev[["sessionID", "connectionTime", "disconnectTime", "cohort"]],
    on="sessionID", validate="many_to_one",
).sort_values(
    ["sessionID", "modifiedAt", "input_order"], kind="stable"
)

# 主队列：到达时最后有效输入应与冻结字段一致。
main = dev.loc[dev["cohort"] == "main_at_arrival"].copy()
visible_at_arrival = updates.loc[
    (updates["cohort"] == "main_at_arrival")
    & (updates["modifiedAt"] <= updates["connectionTime"])
].drop_duplicates("sessionID", keep="last").set_index("sessionID")
assert set(main["sessionID"]) == set(visible_at_arrival.index)
main = main.set_index("sessionID")
assert np.allclose(
    pd.to_numeric(main["initial_kWhRequested"]),
    visible_at_arrival.loc[main.index, "kWhRequested"],
    atol=1e-8, rtol=0,
)
assert (
    aware(main["initial_requestedDeparture"], "initial_requestedDeparture")
    == visible_at_arrival.loc[main.index, "requestedDeparture"]
).all()

# 补充队列：离站事件优先，modifiedAt == disconnectTime 不生效。
late_updates = updates.loc[
    (updates["cohort"] == "supplement_after_submission")
    & (updates["modifiedAt"] >= updates["connectionTime"])
    & (updates["modifiedAt"] < updates["disconnectTime"])
].copy()
late_ids = set(
    dev.loc[
        dev["cohort"] == "supplement_after_submission", "sessionID"
    ]
)
final_targets = late_updates.drop_duplicates("sessionID", keep="last")
assert set(final_targets["sessionID"]) == late_ids
assert len(final_targets) == 1227
final_targets[[
    "sessionID", "modifiedAt", "input_order",
    "kWhRequested", "requestedDeparture",
]].rename(columns={
    "kWhRequested": "evaluation_target_kWh",
    "modifiedAt": "target_source_modifiedAt",
}).to_csv(OUT / "development_supplement_evaluation_targets.csv", index=False)

downward_updates = int(
    late_updates.groupby("sessionID")["kWhRequested"].diff().lt(0).sum()
)

# ---------- 3. 补齐模型规格中的确定规则 ----------
model_path = ROOT / "report/model.md"
model_text = model_path.read_text(encoding="utf-8")
shutil.copy2(model_path, OUT / "model_before.md")

section9 = """## 9. 归一化参考值的确定规则

正式参考值取开发集严格主队列容量受限 Uncontrolled 的实际日费用、
日 EV 峰值中位数；使用开发集全部当地自然日，包含无车日。
若某指标中位数为零，改用该指标严格正值日的中位数；
若无任何正值日，使用 1 USD 或 1 kW，并记录退化标志。

该规则在阶段二确定。实际数值待阶段三实现并验证基线后计算冻结；
随后对所有方法、权重、队列与测试日保持固定，不用测试数据更新。

当前真实数据静态验证使用 C_ref=10 USD、P_ref=50 kW，
仅用于检查数学模型与导出结果，不是正式实验配置。

"""
start = model_text.index("## 9.")
end = model_text.index("## 10.", start)
model_text = model_text[:start] + section9 + model_text[end:]
model_text = model_text.replace(
    "状态：2.1 草稿，待输入接口核对与参数确认；尚未经过数值验证。",
    "状态：阶段二规格、合成验证及开发集严格主队列静态核对已完成；"
    "正式归一化数值按第 9 节在阶段三冻结。",
)
marker = "## 12. 阶段二收尾口径"
if marker not in model_text:
    model_text += """
## 12. 阶段二收尾口径

同刻事件顺序为离站、到达、输入修改；离站时刻的更新不生效。
有效输入同刻冲突按 input_order 升序、最后一条生效。

主队列冻结到达时最后有效请求及预计离站。
补充队列评价目标为连接期间、真实离站前最后有效总请求；
该目标仅供事后评价及完整信息静态参考，不能提前提供给在线控制。
在线控制仍按更新实际生效时刻使用可见目标。

本阶段验证了合成小例子和开发集严格主队列静态模型。
补充队列已核对开发集评价目标提取规则，尚未运行其调度实验。
历史交付量没有参与当前需求、累计交付或缺口归因。
动态信息隔离、实际交付积分与四方法同环境运行属于阶段三。

静态真实连接完整时段数与在线预计离站完整时段数分开统计。
设备与时长导致的缺口均指冻结设备假设及网格下的数学结论。
"""
model_path.write_text(model_text, encoding="utf-8")

# ---------- 4. 保存收尾记录 ----------
summary = {
    "status": "passed",
    "scope": "stage2_model_and_static_validation",
    "synthetic_checks": "16/16",
    "development_main_sessions": 64,
    "main_arrival_target_and_departure_matches": 64,
    "supplement_final_targets_audited": len(final_targets),
    "supplement_downward_update_events": downward_updates,
    "zero_actual_full_slot_sessions": int((detail["full_slots"] == 0).sum()),
    "continuous_device_lower_bound_kWh": float(
        detail["continuous_device_lower_bound_kWh"].sum()
    ),
    "extra_lower_bound_from_complete_slot_grid_kWh": float(
        detail["extra_lower_bound_from_grid_kWh"].sum()
    ),
    "grid_device_lower_bound_kWh": grid_bound,
    "static_U_star_kWh": real["U_star_kWh"],
    "additional_station_capacity_shortfall_kWh": max(
        0.0, real["U_star_kWh"]-grid_bound
    ),
    "formal_normalization_values": "deferred_to_validated_stage3_baseline",
    "formal_experiments_run": False,
}
(OUT / "stage2_acceptance.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

evidence_paths = [
    model_path,
    ROOT / "src/optimization_model.py",
    ROOT / "experiments/run_static.py",
    ROOT / "tests/test_static_lp.py",
    DATA / "user_updates.csv",
    SYN / "validation_summary.json",
    REAL / "static_validation_summary.json",
]
(OUT / "evidence_manifest.json").write_text(
    json.dumps([
        {
            "path": str(p.relative_to(ROOT)),
            "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        }
        for p in evidence_paths
    ], ensure_ascii=False, indent=2),
    encoding="utf-8",
)

acceptance_text = f"""# 阶段二验收记录

证据目录：`{OUT.relative_to(ROOT)}`。

- 2.1：规格、单位、信息集、两类队列目标、终端与归一化规则已明确。
- 2.2：静态双层 LP、独立入口、两层日志与逐车逐时段计划已生成。
- 2.3：独立合成检查 16/16 通过。
- 开发集严格主队列：64 车、4704 时段；两层均确认最优。
- 第一层缺口：{real["U_star_kWh"]:.6f} kWh，与单车完整时段下界一致。
- 缺口归因见 `session_shortfall_audit.csv`。
- 开发集补充队列 1227 条评价目标已审计，未运行其调度实验。
- 正式归一化数值待阶段三验证基线后按冻结规则确定。
- 尚未开展动态信息隔离、能量执行积分或四方法实验。

允许进入阶段三实现；本记录不代表在线效果或原网络电气可行性。
"""
report_path = ROOT / "report/stage2_acceptance.md"
if report_path.exists():
    shutil.copy2(report_path, OUT / "stage2_acceptance_before.md")
report_path.write_text(acceptance_text, encoding="utf-8")
(OUT / "stage2_acceptance.md").write_text(acceptance_text, encoding="utf-8")

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("模型规格：", model_path)
print("验收记录：", report_path)
print("逐车缺口审计：", OUT / "session_shortfall_audit.csv")

本次审计目录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/results/logs/stage2_acceptance_20261112T071653Z_0b9320
{
  "status": "passed",
  "scope": "stage2_model_and_static_validation",
  "synthetic_checks": "16/16",
  "development_main_sessions": 64,
  "main_arrival_target_and_departure_matches": 64,
  "supplement_final_targets_audited": 1227,
  "supplement_downward_update_events": 13,
  "zero_actual_full_slot_sessions": 3,
  "continuous_device_lower_bound_kWh": 156.75622222222225,
  "extra_lower_bound_from_complete_slot_grid_kWh": 30.289777777777765,
  "grid_device_lower_bound_kWh": 187.046,
  "static_U_star_kWh": 187.04599999999996,
  "additional_station_capacity_shortfall_kWh": 0.0,
  "formal_normalization_values": "deferred_to_validated_stage3_baseline",
  "formal_experiments_run": false
}
模型规格： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/report/model.md
验收记录： /home/jovyan/work/EV_Rolling_Horizon_Charging_Optimization/report/stage2_acceptance.md
逐车缺口审计： /ho